# AI in Industry — Lab

`notebook v9` — Step 1 tells you if this copy is out of date.

**Nothing installs on your computer.** Everything runs on Google's servers; your laptop only needs a browser.

---

## First — make your own copy, then close the old tab

Click **File → Save a copy in Drive**.

A new tab opens, titled **`Copy of lab.ipynb`**.

> **Now close the tab you were in before.** You have two tabs open and they look almost identical. The old one is not yours — nothing you type there is saved, and it is easy to spend a whole session in the wrong tab.
>
> Check the title at the top left reads **`Copy of lab.ipynb`** before going on.

---

## Then — get a free API key

Open **[console.groq.com](https://console.groq.com/keys)** → sign in with Google → **API Keys** → **Create API Key** → name it `lab` → **Submit**.

Copy the key. It starts `gsk_` and is **shown only once**. Lose it and you just make another; it is free.

---

## Then — store the key in Colab

Do **not** paste your key into a code cell.

1. On the **far left edge**, click the **🔑 key icon**. The **Secrets** panel opens.
2. Click **+ Add new secret**.
3. Fill in the middle two columns, then switch on the first:

| Column | What to do |
|---|---|
| **Name** | Type `LLM_API_KEY` — all capitals, two underscores |
| **Value** | Paste your `gsk_...` key |
| **Notebook access** | **Click the toggle so it turns on** |

### The two things that go wrong here

**The Notebook access toggle starts OFF** — a grey ✕ until you click it. Leave it off and the notebook cannot see your key, even though the secret looks saved. This is the most common failure.

**The Name box is narrow and hides the end of what you typed.** It can show `LLM_API` whether or not you typed it all. Click in and press `End` to check it reads `LLM_API_KEY` exactly.

---

Run **Step 1** and **Step 2** below. Then open whichever lab step you want — the links are at the bottom of this notebook.

## Step 1 — Download the lab files

**Run this again whenever you like.** It deletes the old copy and downloads fresh, so you are never running yesterday's code against today's instructions.

Re-run it if something looks broken, or if the lecturer says there is a fix.

In [ ]:
#@title STEP 1 - Download the lab files { display-mode: "form" }
# Safe to re-run at any time: it deletes the old copy and downloads
# fresh, so you can never be running yesterday's code.
import os, sys, pathlib, subprocess, time

NOTEBOOK_VERSION = 9
REPO, NAME, SUB = "coolMukul/rough", "rough", "ai-lab"
BASE = "/content" if pathlib.Path("/content").exists() else "."
LAB = f"{BASE}/{NAME}/{SUB}"

# Deleting a folder you are standing in leaves every later ! command failing
# at getcwd, which reads as a missing file. Move somewhere valid first.
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)


def ok(m):
    print(f"      OK - {m}", flush=True)


def info(m):
    print(f"      {m}", flush=True)


def die(m, fix):
    print(f"\n      FAILED - {m}\n")
    for line in fix.strip().splitlines():
        print("      " + line.strip())
    print("\n      Stuck? Carry on to the next step anyway - every step of")
    print("      the lab sets itself up, so one failure does not strand you.\n")
    raise SystemExit(1)


def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).strip()


def secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return os.environ.get(name, "").strip()

print("=" * 64)
print("  STEP 1 - Downloading the lab files")
print("=" * 64)

os.chdir(BASE)

if pathlib.Path(NAME).exists():
    info("removing the previous copy")
    run(f"rm -rf {NAME}")

info(f"downloading {REPO}")
code, out = run(f"git clone --depth 1 https://github.com/{REPO}.git {NAME}")
if code != 0:
    die(out, """
        Check your internet connection and run this cell again.
        If it keeps failing, tell the lecturer - the address may have changed.
        """)

os.chdir(LAB)
info(f"working directory: {os.getcwd()}")

missing = [f for f in ("requirements.txt", "verify_setup.py", "labkit.py",
                       "data/chunks.json") if not pathlib.Path(f).exists()]
if missing:
    die("missing after download: " + ", ".join(missing),
        "Run this cell again. If it still fails, tell the lecturer.")
ok("all lab files present")

try:
    latest = int(pathlib.Path("lab-version.txt").read_text().strip())
    if NOTEBOOK_VERSION < latest:
        info(f"NOTE: this notebook is v{NOTEBOOK_VERSION}, latest is v{latest}")
        info("      The code you just downloaded is current, so carry on.")
        info("      For the newest instructions take a fresh copy from:")
        info(f"      https://colab.research.google.com/github/{REPO}/blob/main/{SUB}/lab.ipynb")
    else:
        ok(f"notebook v{NOTEBOOK_VERSION} (current)")
except Exception:
    pass

print("\n  Done. Now run STEP 2.")

## Step 2 — Install packages and check your key

Run after Step 1. **2–3 minutes the first time** — it downloads a small language model. It will look frozen. It is not.

Ends with `READY` when everything is in place.

In [ ]:
#@title STEP 2 - Install packages and check your key { display-mode: "form" }
# Run after STEP 1. Slow the first time - it downloads a small model.
import os, sys, pathlib, subprocess, time

NOTEBOOK_VERSION = 9
REPO, NAME, SUB = "coolMukul/rough", "rough", "ai-lab"
BASE = "/content" if pathlib.Path("/content").exists() else "."
LAB = f"{BASE}/{NAME}/{SUB}"

# Deleting a folder you are standing in leaves every later ! command failing
# at getcwd, which reads as a missing file. Move somewhere valid first.
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)


def ok(m):
    print(f"      OK - {m}", flush=True)


def info(m):
    print(f"      {m}", flush=True)


def die(m, fix):
    print(f"\n      FAILED - {m}\n")
    for line in fix.strip().splitlines():
        print("      " + line.strip())
    print("\n      Stuck? Carry on to the next step anyway - every step of")
    print("      the lab sets itself up, so one failure does not strand you.\n")
    raise SystemExit(1)


def run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).strip()


def secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return os.environ.get(name, "").strip()

t0 = time.time()
print("=" * 64)
print("  STEP 2 - Installing and checking")
print("=" * 64)

if not pathlib.Path(LAB).exists():
    die("the lab files are not here yet", "Run STEP 1 first, then this cell.")
os.chdir(LAB)

print("\n[1/3] Installing packages - SLOW the first time, please wait")
code, out = run(f"{sys.executable} -m pip install -q -r requirements.txt")
if code != 0:
    print(out[-1200:])
    die("pip install failed",
        "Runtime -> Restart session, then run STEP 1 and STEP 2 again.")
ok("installed")

print("\n[2/3] Looking for your API key")
key = secret("LLM_API_KEY")
if not key:
    die("no key found in Colab secrets", """
        1. Click the key icon on the far left edge
        2. Click '+ Add new secret'
        3. Name it exactly:  LLM_API_KEY
           The name box is narrow and hides the end of long text.
           Click into it and press End to check the whole name is there.
        4. Paste your key from  https://console.groq.com/keys
        5. Turn ON the 'Notebook access' toggle - it starts OFF and
           shows a grey X. This is the most common mistake.
        6. Run this cell again
        """)
if key != key.strip() or len(key) < 20:
    die("the key looks malformed (too short, or spaces around it)",
        "Re-copy it from the Groq console and update the secret.")
os.environ["LLM_API_KEY"] = key
ok(f"found, ending ...{key[-4:]}")

print("\n[3/3] Running the full check")
print()
code, out = run(f"{sys.executable} verify_setup.py")
print(out)

print("\n" + "=" * 64)
if code == 0:
    print(f"  READY. Took {time.time() - t0:.0f} seconds.")
else:
    print("  Some checks failed - read the FAIL lines above for the fix.")
    print("  You can still carry on: every step of the lab sets itself up.")
print("=" * 64)

### If a check says FAIL

The message underneath tells you the fix. The two most common:

| Message | Fix |
|---|---|
| `no API key found` | Secret must be named exactly `LLM_API_KEY`, and **Notebook access** must be ON. |
| `401 unauthorised` | The key is wrong, or has a space at either end. Re-copy it from Groq. |

**Do not stop and debug during the session.** Open the next step's notebook instead — each one sets itself up from scratch, so you will be caught up automatically.

## The lab steps

**Each step is its own notebook.** Click a link and it opens in Colab, ready to run — the code is there for you to read, change, and run again.

| # | Open it | What you take away | Cost |
|---|---|---|---|
| **1** | [The model has no memory](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_01.ipynb) | There is no memory on the server. Everything that feels like a conversation is a Python list your own code re-sends every time. | 3 calls |
| **2** | [Without your data, it invents](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_02.ipynb) | A wrong answer arrives in exactly the same confident tone as a right one. You cannot tell them apart by reading. | 1 call |
| **3** | [Load the regulations](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_03.ipynb) | How a document gets turned into something searchable, and why each chunk carries its section name. | **free** |
| **4** | [Find the right clause](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_04.ipynb) | Retrieval is ordinary search — the DBMS material you already know, with a different relevance score. | **free** |
| **5** | [This is RAG](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_05.ipynb) | That is all RAG is. Retrieve, stuff into the prompt, instruct the model to stay inside it. | 1 call |
| **6** | [Break it](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_06.ipynb) | Most of the time "the AI is wrong", the retrieval was wrong. This is the most useful debugging instinct in the session. | 3 calls |
| **7** | [Match on meaning, not words](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_07.ipynb) | What embeddings are for — and that swapping in a fancier technique does not automatically make a system better. | 1 call |
| **8** | [A workflow](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_08.ipynb) | How a model becomes part of ordinary software: it turns prose into JSON that an `if` statement can act on. | 4 calls |
| **11** | [Measure it](https://colab.research.google.com/github/coolMukul/rough/blob/main/ai-lab/notebooks/step_11.ipynb) | How to tell whether a change made things better, with a number instead of an opinion. Almost nobody does this. | 30 calls |

**Steps 3 and 4 make no API calls at all.** Start with those — explore your own regulations as much as you like without spending any quota.

### Two things worth knowing

**You do not have to save these.** Open, run, edit, close. Save a copy to Drive only for the one you want to keep working on afterwards.

**Each notebook needs the secret switched on.** Your `LLM_API_KEY` is stored once against your Google account, but Colab asks *per notebook* whether that notebook may read it. So in each one: key icon on the left, find `LLM_API_KEY`, turn **Notebook access** on. Two clicks — you are not making a new key.

### If a notebook gives you trouble

**Skip it and open the next one.** Every notebook sets itself up from scratch, so nothing you missed will leave you behind.